# Transformer From Scratch in PyTorch

This notebook implements core transformer components from scratch using PyTorch.

Implemented components:
- Linear layer
- Embedding layer
- SiLU activation
- RMS Normalization
- Self Attention
- Multi-Head Attention with KV Cache
- MLP block
- Transformer block
- Cross Entropy Loss
- Adam Optimizer

`Goal`:
Understand how modern transformer architectures work internally.

In [22]:
import torch
from torch.nn import Module, ModuleList, Parameter, Buffer
import tiktoken
import math
import os
import re
import numpy as np
print('Hello World!')

Hello World!


# Linear layer with no bias term. The parameters of the layer are stored in a `.weight Parameter`.

In [23]:
class Linear(Module):
    def __init__(self, in_dim, out_dim):
        """ 
        Initialize a linear layer without a bias term.
        Inputs:
            in_dim : int - input feature dim
            out_dim : int - output feature dim
        """
        super().__init__()
        self.weight = Parameter(torch.randn(out_dim,in_dim)*math.sqrt(2/in_dim)) ### `He Initiation`
    def forward(self,X):
        """
        Apply linear layer to one or more input vectors.
        Input:
            X : torch.Tensor[float] (... x in_dim) - input tensor
        Output:
            torch.Tensor[float] (... x out_dim) - transformed tensor
        """
        return X@self.weight.T

In [24]:
layer = Linear(4,3)
x = torch.randn(2,4)
y = layer(x)
print(y.shape)

torch.Size([2, 3])


# Initialize an embedding table over a fixed vocabulary.

Embedding convert token IDs into dense vectors.

In [25]:
class Embedding(Module):
    def __init__(self, num_tokens, dim):
        """
        Inputs:
            num_tokens : int - vocabulary size (i.e. how many token ids your model knows)
            dim : int - embedding dimention
        """
        super().__init__()
        ### weight is a matrix of embeddings for each of the V tokens (num_tokens)
        self.weight = Parameter(torch.randn(num_tokens,dim))
    def forward(self, Y):
        """
        Look up embeddings for an integer tensor of token ids
        Input:
            Y : torch.Tensor[int] (...) - tensor of token indices in [0, num_tokens)
        Output:
            torch.tensor[float] (... x dim) - embedding vectors for each token id
        """
        return self.weight[Y] 

In [26]:
embed = Embedding(10,5)
tokens = torch.tensor([2,0,1])
output = embed(tokens)
print(output)

tensor([[ 0.8136, -1.3106, -1.7470, -0.8099,  1.1419],
        [ 0.5150, -0.6472,  0.4676, -0.1255, -2.4544],
        [-0.4514, -1.4087,  0.7052, -0.2305, -0.8146]],
       grad_fn=<IndexBackward0>)


# SiLU Activation: Apply the SiLU nonlinearity elementwise.

Unlike ReLU, SiLU is smooth and differentiable everywhere.

### ReLU Activation

The nonlinearity used is the **ReLU (Rectified Linear Unit)** function:

$$
\sigma(x) = \max(0, x)
$$

---
### SiLU Activation

The nonlinearity used is the **SiLU (Sigmoid Linear Unit)** function:

$$
\text{SiLU}(x) = x \cdot \sigma(x)
$$

where the sigmoid function is

$$
\sigma(x) = \frac{1}{1 + e^{-x}}
$$


---

In [27]:
def silu(x):
    """
    Input:
        x : torch.Tensor[float] (...) - input tensor
    Output:
        torch.Tensor[float] (...) - tensor after applying SiLU
    """
    return x*torch.sigmoid(x)

# Apply RMS normalization along the final dimension of the input

$$
\mathrm{RMS}(x)
=
\sqrt{
\frac{1}{d}
\sum_{i=1}^{d} x_i^2
+ \varepsilon
}
$$

In [28]:
def rms_norm(X, eps=1e-5):
    """
    Inputs:
        X : torch.Tensor[float] (... x dim) - input tensor
        eps: float - munerical stability constant
    Output:
        torch.Tensor[float] (... x dim) - RMS-normalized tensor
    """
    rms_X = torch.sqrt(torch.mean(X**2,dim=-1,keepdim=True)+eps)
    return X/rms_X

# Self Attention: is a way mixing together infos across diff rows of the matrix

Apply scaled dot-product attention, optionally with an additive mask.

$$
\text{Attention}(Q, K, V)
=
\mathrm{softmax}\left(
\frac{QK^\top}{\sqrt{d}}
+ \text{mask}
\right)V
$$


In [29]:
def self_attention(Q,K,V, mask=None):
    """
    Inputs:
        Q: torch.Tensor[float] (... x query_len x d) - query tensor
        K: torch.Tensor[float] (... x key_len x d) - key tensor
        V: torch.Tensor[float] (... x key_len x d_v) - value tensor
        mask: torch.Tensor[float] (... x query_len x key_len) or None - additive attention mask
    Output:
        torch.Tensor[float] (... x query_len x d_v) - attention output tensor
    """
    d = Q.shape[-1]
    scores = (Q@K.transpose(-1,-2))/math.sqrt(d)
    if mask is not None: ### masking stops mixing among past and future words.
        scores += mask
    return torch.softmax(scores, dim=-1)@V

# Multi-Head Attention `KV Cache` (efficient LLM Inference)

$$
Y_{T+1}
=
\mathrm{softmax}\left(
\frac{
Q_{T+1}
\begin{bmatrix}
K \\
K_{T+1}
\end{bmatrix}^\top
}{
\sqrt{d}
}
\right)
\begin{bmatrix}
V \\
V_{T+1}
\end{bmatrix}
$$

In [65]:
class MultiHeadAttentionKVCache(Module):
    def __init__(self, dim, n_heads, max_cache_size):
        """
        Initialize a multi-head self-attention layer with KV cache buffers.
        
        Inputs:
            dim: int - total embedding dimention
            n_heads: int - number of attention heads
            max_cache_size: int - maximum sequence length stored in the cache
        """
        super().__init__()
        self.n_heads = n_heads
        self.each_head_dim = dim // n_heads
        assert dim % n_heads == 0

        self.wq = Linear(dim,dim)
        self.wk = Linear(dim,dim)
        self.wv = Linear(dim,dim)
        self.wp = Linear(dim,dim)
        
        ### KV cache (batch_size = 1)
        self.k_cache = Buffer(torch.zeros(1, max_cache_size, dim))
        self.v_cache = Buffer(torch.zeros(1, max_cache_size, dim))

    def forward(self, X, mask=None, seq_pos=0, use_kv_cache=False):
        """
        Apply multi-head self-attention, optionally updating and using the KV cache.

        Inputs:
            X: torch.Tensor[float] (batch_size x seq_len x dim) - input sequence embedding (this X is := self.weight[Y] after embedding)
            mask: torch.Tensor[float] (seq_len x total_len) or None - additive attention mask
            seq_pos: int - starting sequence position for cached tokens
            use_kv_cache : bool - wheather to update and use the KV cache
        Output:
            torch.Tensor[float] (batch_size x seq_len x dim) - attention output
        """
        B, N, D = X.shape # batch_size x seq_len (number of different elements/words in sequence/text) x embedding_dim
        Q, K, V = self.wq(X), self.wk(X), self.wv(X)

        if use_kv_cache:
            ### populate and update cache
            self.k_cache[:, seq_pos:seq_pos+N, :] = K
            self.v_cache[:, seq_pos:seq_pos+N, :] = V
            ###  prepare the entire updated cache to be used in self_attention()
            K = self.k_cache[:, :seq_pos+N, :]
            V = self.v_cache[:, :seq_pos+N, :]

        ### Multihead Attention: (B, N, D) => (B, N, n_heads, D/heads) = (B,N,self.n_heada, self.each_head_dim)
        Q = Q.reshape(B, N, self.n_heads, self.each_head_dim).transpose(1,2) # .transpose(1,2) as attention is computed per head.
        K = K.reshape(B, K.shape[1], self.n_heads, self.each_head_dim).transpose(1,2) #seq_len of K,V changes as we update cache (i.e. new len = .shape[1]) 
        V = V.reshape(B, V.shape[1], self.n_heads, self.each_head_dim).transpose(1,2) #seq_len of K,V changes as we update cache (i.e. new len = .shape[1])

        Y = self_attention(Q,K,V,mask)
        Y = Y.transpose(1, 2).reshape(B, N, D)
        return self.wp(Y)

### MultiHeadAttentionKVCache for post-training

In [30]:
class MultiHeadAttentionKVCache(Module):
    def __init__(self, dim,n_heads,max_cache_size, max_cache_batches=64):
        """
        Inputs:
            dim: int - total embedding dimension
            n_heads : int - number of attention heads
            max_cache_size : int - max sequence length stored in the cache
            max_chace_batches : int - max batch size for cache
        """
        super().__init__()
        self.max_cache_size = max_cache_size
        self.max_cache_batches = max_cache_batches
        self.wq,self.wk,self.wv,self.wp = [Linear(dim,dim) for _ in range(4)]
        ### multihead infos
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        ###
        self.k_cache = Buffer(torch.zeros(max_cache_batches,max_cache_size,dim))
        self.v_cache = Buffer(torch.zeros(max_cache_batches,max_cache_size,dim))
    def forward(self, X, mask=None,seq_pos=0,use_kv_cache=False):
        """
        Apply multi-head self-attention, optionally updating and using the KV cache.
        Inputs:
            X : torch.Tensor[float] (batch_size x seq_len x dim) - input sequence embeddings
            mask : torch.Tensor[float] (seq_len x total_len) or None - additive mask
            seq_pos : int - starting sequence position for cached tokens
            use_kv_cache : bool - whether to update and use the KV cache
        Output:
            torch.Tensor[float] (batch_size x seq_len x dim) - attention output
        """
        B,N,D = X.shape
        if use_kv_cache:
            Q,K_new,V_new = self.wq(X),self.wk(X),self.wv(X)
            ### populate or append new keys/values
            K_new = self.k_cache[:B,seq_pos:seq_pos+N,:]
            V_new = self.v_cache[:B,seq_pos:seq_pos+N,:]
            ### Here, K,V include everything so far, but past tokens come from cache and new tokens were appended.
            K = self.k_cache[:B,:seq_pos+N,:]
            V = self.v_cache[:B,:seq_pos+N,:]
        else:
            Q,K,V = self.wq(X),self.wk(X),self.wv(X)

        Q = Q.reshape(B,N,self.n_heads,self.head_dim).transpose(1,2)
        K = K.reshape(B,K.shape[1],self.n_heads,self.head_dim).transpose(1,2)
        V = V.reshape(B,V.shape[1],self.n_heads,self.head_dim).transpose(1,2)

        Y = self_attention(Q,K,V,mask)
        Y = Y.transpose(1,2).reshape(B,N,D)
        return self.wp(Y)


# Feed Forward Network (MLP)

Transformers use position-wise feed forward layers after attention

In [45]:
class MLP(Module):
    def __init__(self, dim, ffn_dim):
        """
        Initialize the gated feed-forward network used in the transformer block.
        Inputs:
            dim : int - model dimension
            ffn_dim : int - hidden feed-forward dimension
        """
        super().__init__()
        self.w1 = Linear(dim,ffn_dim)
        self.w2 = Linear(ffn_dim,dim)
    def forward(self, X):
        """
        Apply the gated feed-forward network to the input tensor.
        Input:
            X : torch.Tensor[float] (... x dim) - input tensor
        Output:
            torch.Tensor[float] (... x dim) transformed tensor
        """
        return self.w2(silu(self.w1(X)))

# Transformer Layer

A transformer layer consists of:

- Multi-Head Self-Attention
- MLP (Feed-Forward Network)
- Normalization
- Residual Connections

---

### Attention Block

Given an input tensor \(X\),

$$
Z
=
X
+
\mathrm{MultiHeadAttention}(
\mathrm{Norm}(X)
)
$$

---

### MLP Block

The final output is computed as

$$
Y
=
Z
+
\mathrm{MLP}(
\mathrm{Norm}(Z)
)
$$

Again, a residual connection is applied after the MLP block.


In [32]:
class TransformerBlock(Module):
    def __init__(self, dim, n_heads, ffn_dim, max_cache_size):
        """
        Inputs:
            dim : int - model dimension
            n_heads : int - number of attention heads
            ffn_dim : int - hidden feed-forward dimension
            max_cache_size : int - maximum sequence length stored in the attention cache
        """
        super().__init__()
        self.attn = MultiHeadAttentionKVCache(dim,n_heads,max_cache_size)
        self.mlp = MLP(dim,ffn_dim)
    def forward(self, X, mask=None, seq_pos=0, use_kv_cacche=False):
        """
        Apply one trasformer block with residual connections.
        Inputs:
            X : torch.Tensor[float] (batch size x seq_len x dim) - input sequence embeddings
            mask : torch.Tensor[float] (seq_len x total_len) or None - additive attention mask
            seq_pos : int - starting sequence position for cached tokens
            use_kv_cache : bool - whether to update and use the attention cache
        Output:
            torch.Tensor[float] (batch_size x seq_len x dim) - transformed sequence embeddings
        """
        z = X + self.attn(rms_norm(X),mask,seq_pos,use_kv_cacche)
        y = z + self.mlp(rms_norm(z))
        return z

# LLM : Initialize the simplified `Llama 3` model. 

Official resources:
- [Meta Llama 3 Official Page](https://llama.meta.com/llama3/?utm_source=chatgpt.com)
- [Meta AI Llama 3 Announcement](https://ai.meta.com/blog/meta-llama-3/?utm_source=chatgpt.com)
- [Llama 3 Research Paper](https://ai.meta.com/research/publications/the-llama-3-herd-of-models/?utm_source=chatgpt.com)

# Transformer = Embedding + (N x) Transformer Layers + normalization + output linear layer

* mask: a PyTorch Buffer containing a (max_seq_len, max_seq_len) strictly upper triangular mask (negative infinity in strictly upper diagonal entries, zero elsewhere). You will pass subsets of this mask to the transformer layers based upon the actual sequence position of the token inputs.


\begin{bmatrix}
0 & -\infty & -\infty & -\infty \\
0 & 0 & -\infty & -\infty \\
0 & 0 & 0 & -\infty \\
0 & 0 & 0 & 0
\end{bmatrix}


In [43]:
class LLM(Module):
    def __init__(self, num_tokens, dim, n_heads, max_seq_len, ffn_dim, num_layers):
        """
        Inputs:
            num_tokens : int - vocabulary size (i.e. how many token ids your model knows)
            dim : int - model dimension
            n_heads : int - number of attention heads per layer
            max_seq_len : int - max supported sequence length (i.e. how many tokens model can process @ once)
            ffn_dim : int - hidden feed-forward dimension in each block
            num_layers : int - number of transformer blocks
        """
        super().__init__()
        ### Embedd your vocabulary
        self.embedding = Embedding(num_tokens,dim) 
        ### Fix the order invariance by position embedding
        self.pos_embedding = Parameter(torch.randn(max_seq_len,dim))
        ### a Pytorch Modulelist of length num_layers, where each element is a TransformerBlock
        self.layers = ModuleList()
        for i in range(num_layers):
            self.layers.append(TransformerBlock(dim,n_heads,ffn_dim,max_seq_len))
        self.output = Linear(dim,num_tokens)
        ### Fix past-future relation by masking
        mask = torch.triu(torch.full((max_seq_len, max_seq_len), float('-inf')), diagonal=1)
        self.mask = Buffer(mask)
    
    def forward(self, tokens, seq_pos=0, use_kv_cache=False):
        """
        Apply the full LLM to a batch of token sequences
        Inputs:
            tokens : torch.Tensor[int] (batch_size x seq_len) - input token ids (each row is a sentence)
            seq_pos : int - startig sequence position for positional embedding and cached attention
            use_kv_cache : bool - whether to update and use the attention cache
        Output:
            torch.Tensor[float] (batch_size x seq_len x num_tokens)
        """
        ### Embedd your input token ids
        X = self.embedding(tokens)
        N = X.shape[1]
        ### Add position embedding to your inputs
        X = X + self.pos_embedding[seq_pos:seq_pos+N]
        mask = self.mask[seq_pos:seq_pos+N, :seq_pos+N]
        ### Apply TransformerBlock for each layer
        for layer in self.layers:
            X = layer(X, mask, seq_pos, use_kv_cache)
        X = rms_norm(X)
        logits = self.output(X)
        return logits

# Transformer / LLM Forward Pipeline

The model architecture is:

$$
\text{tokens}
\rightarrow
\text{Embedding}
\rightarrow
\text{Positional Embedding}
\rightarrow
N \times \text{Transformer Blocks}
\rightarrow
\text{RMSNorm}
\rightarrow
\text{Linear Output Layer}
\rightarrow
\text{logits}
$$

---

# Forward Pass

Given input token IDs:

$$
\text{tokens}
\in
\mathbb{Z}^{(\text{batch\_size} \times \text{seq\_len})}
$$

the model computes:

---

## 1. Token Embedding

Convert token IDs into dense vectors:

$$
X
=
\mathrm{Embedding}(\text{tokens})
$$

Shape:

$$
(\text{batch\_size},\ \text{seq\_len},\ \text{dim})
$$

---

## 2. Add Positional Embeddings

Add positional information:

$$
X
=
X
+
P[\text{seq\_pos}:\text{seq\_pos}+N]
$$

---

## 3. Create Attention Mask

Construct the attention mask:

$$
M
=
\text{mask}[
\text{seq\_pos}:\text{seq\_pos}+N,
:
\text{seq\_pos}+N
]
$$

This prevents tokens from attending to future positions.

---

## 4. Apply Transformer Layers

For each transformer block:

$$
X
=
\mathrm{TransformerBlock}(X, M)
$$

Each block contains:

$$
\text{Transformer Block}
=
\text{Self-Attention}
+
\text{MLP}
+
\text{Normalization}
+
\text{Residual Connections}
$$

---

## 5. Final Normalization

Apply RMS normalization:

$$
X
=
\mathrm{RMSNorm}(X)
$$

---

## 6. Output Projection

Project `hidden states` into `vocabulary logits`:

$$
\text{logits}
=
XW_{\text{out}}
$$

Output shape:

$$
(\text{batch\_size},\ \text{seq\_len},\ \text{num\_tokens})
$$

where each position contains scores for every token in the vocabulary.

---